# Interactive PaCMAP (Hover + Zoom + Audio Snippets)

Standalone interactive notebook with click-to-play snippets and top-25 cosine-similar tracks.


In [2]:
# Interactive PaCMAP with click-to-play snippets and top-25 similarity panel.
import csv
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import webbrowser

from djprojectexploration.audio_snippets import ensure_cached_snippet

try:
    import pacmap
except ImportError as exc:
    raise ImportError('PaCMAP is not installed. Install with: pip install pacmap') from exc

def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / 'data').exists() and (c / 'notebooks').exists():
            return c
    return cwd

PROJECT_ROOT = detect_project_root()
NPZ_EMBEDDINGS_FILE = PROJECT_ROOT / 'data' / 'maest_embeddings' / 'dataset_tracks.npz'
GENRE_TAGS_CSV = PROJECT_ROOT / 'data' / 'exports' / 'dataset_tracks.csv'

# Optional manual genre collapsing. Keys are matched case-insensitively against raw CSV genres.
GENRE_ALIAS_MAP: dict[str, str] = {
    'progressive trance': 'Trance',
    'uplifting trance': 'Trance',
    'tech trance': 'Trance',
    'melodic techno': 'Progressive',
    'progressive house': 'Progressive',
    'deep house': 'Progressive',
    'tech house': 'House',
    'bass house': 'House',
    'uk garage': 'House',
    'bass house]': 'Electro',
    'future house': 'Electro',
    'electro house': 'Electro',
    'future rave': 'Electro',
    'big room': 'Electro',
    'hip hop': 'Rap',
    'electronica': 'Electronic',
    'experimental electronic': 'Electronic',
}
# Keep only these final identities (set to None to keep everything after alias mapping).
GENRE_WHITELIST: set[str] | None = None
OTHER_GENRE_LABEL = 'Other'

def canonicalize_genre(raw_genre: str) -> str:
    genre = (raw_genre or '').strip() or 'Unknown'
    canonical = GENRE_ALIAS_MAP.get(genre.casefold(), genre)
    if GENRE_WHITELIST is None:
        return canonical
    allowed = {g.casefold() for g in GENRE_WHITELIST}
    return canonical if canonical.casefold() in allowed else OTHER_GENRE_LABEL

npz_path = NPZ_EMBEDDINGS_FILE.expanduser()
csv_path = GENRE_TAGS_CSV.expanduser()
if not npz_path.exists():
    raise FileNotFoundError(f'NPZ embeddings file not found: {npz_path}')
if not csv_path.exists():
    raise FileNotFoundError(f'Genre CSV file not found: {csv_path}')

npz_payload = np.load(npz_path)
if 'embeddings' not in npz_payload.files or 'filenames' not in npz_payload.files:
    raise KeyError('NPZ must contain `embeddings` and `filenames` arrays.')

emb = np.asarray(npz_payload['embeddings'], dtype=np.float32)
filenames = [str(x) for x in npz_payload['filenames']]
if emb.ndim != 2 or emb.shape[0] < 2:
    raise ValueError('Need a 2D embedding matrix with at least 2 tracks.')
if len(filenames) != emb.shape[0]:
    raise ValueError('`filenames` length must match embeddings rows.')

def norm_token(value: str) -> str:
    return Path(str(value).strip()).name.lower()

def row_token(row: dict) -> str | None:
    for key in ('mp3_name', 'filename', 'filepath', 'location'):
        val = (row.get(key) or '').strip()
        if val:
            return norm_token(val)
    return None

def resolve_audio_path(row: dict) -> Path | None:
    for key in ('filepath', 'location'):
        val = (row.get(key) or '').strip()
        if val:
            p = Path(val).expanduser()
            if p.exists():
                return p.resolve()

    mp3_name = (row.get('mp3_name') or row.get('filename') or '').strip()
    if not mp3_name:
        return None

    rel = Path(mp3_name)
    candidates = [
        csv_path.parent / rel,
        PROJECT_ROOT / rel,
        PROJECT_ROOT / 'music' / rel.name,
        PROJECT_ROOT / 'music' / 'ara-mix' / rel.name,
        PROJECT_ROOT / 'music' / 'aries-mix' / rel.name,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None

SNIPPET_SECONDS = 8.0
SNIPPET_MIDDLE_FRACTION = 0.66
SNIPPET_HOP_SECONDS = 0.25
SNIPPET_CACHE_OVERWRITE = False
SNIPPET_CACHE_DIR = PROJECT_ROOT / 'data' / 'snippets' / npz_path.stem
HTML_EXPORT_DIR = PROJECT_ROOT / 'data' / 'exports'
INTERACTIVE_HTML_PATH = HTML_EXPORT_DIR / f'{npz_path.stem}_interactive_pacmap.html'

csv_by_token: dict[str, dict] = {}
with csv_path.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        token = row_token(row)
        if token and token not in csv_by_token:
            csv_by_token[token] = row

records = []
for i, filename in enumerate(filenames):
    token = norm_token(filename)
    row = csv_by_token.get(token, {})

    title = (row.get('title') or row.get('name') or Path(filename).stem).strip()
    artist = (row.get('artists') or row.get('artist') or '').strip()
    raw_genre = (row.get('genre') or 'Unknown').strip() or 'Unknown'
    genre = canonicalize_genre(raw_genre)
    bpm = (row.get('bpm') or '').strip()
    key = (row.get('key') or '').strip()
    album = (row.get('album') or '').strip()
    track_number = (row.get('track_number') or row.get('#') or '').strip()
    audio_path = resolve_audio_path(row)

    try:
        snippet = ensure_cached_snippet(
            audio_path=audio_path,
            output_dir=SNIPPET_CACHE_DIR,
            key=filename,
            snippet_seconds=SNIPPET_SECONDS,
            middle_fraction=SNIPPET_MIDDLE_FRACTION,
            hop_seconds=SNIPPET_HOP_SECONDS,
            overwrite=SNIPPET_CACHE_OVERWRITE,
            project_root=PROJECT_ROOT,
        )
    except Exception:
        snippet = None

    if snippet is None:
        snippet_uri, start_s, end_s, snippet_rms = '', 0.0, 0.0, 0.0
        snippet_path = ''
    else:
        snippet_file = snippet.snippet_path.resolve()
        start_s = snippet.start_seconds
        end_s = snippet.end_seconds
        snippet_rms = snippet.rms
        snippet_path = str(snippet_file)
        try:
            snippet_uri = snippet_file.relative_to(INTERACTIVE_HTML_PATH.parent.resolve()).as_posix()
        except ValueError:
            snippet_uri = snippet_file.as_uri()

    records.append({
        'idx': i,
        'filename': filename,
        'title': title,
        'artist': artist,
        'genre': genre,
        'bpm': bpm,
        'key': key,
        'album': album,
        'track_number': track_number,
        'audio_path': '' if audio_path is None else str(audio_path),
        'snippet_uri': snippet_uri,
        'snippet_start': start_s,
        'snippet_end': end_s,
        'snippet_rms': snippet_rms,
        'snippet_path': snippet_path,
    })

# Similarity + distribution probabilities.
norms = np.linalg.norm(emb, axis=1, keepdims=True)
norms = np.where(norms == 0.0, 1.0, norms)
emb_norm = emb / norms
cosine_similarity = np.clip(emb_norm @ emb_norm.T, -1.0, 1.0).astype(np.float32)
cosine_distance = np.clip(1.0 - cosine_similarity, 0.0, 2.0).astype(np.float32)

temperature = 0.08
n_tracks = emb.shape[0]
similarity_payload: dict[str, dict[str, object]] = {}

for i in range(n_tracks):
    logits = cosine_similarity[i].astype(np.float64).copy()
    logits[i] = -np.inf

    finite = np.isfinite(logits)
    probs = np.zeros(n_tracks, dtype=np.float64)
    if np.any(finite):
        x = logits[finite] / max(temperature, 1e-6)
        x = x - np.max(x)
        exp_x = np.exp(x)
        denom = np.sum(exp_x)
        if denom > 0.0:
            probs[finite] = exp_x / denom

    order = np.argsort(cosine_distance[i])
    top_idx = [int(j) for j in order if int(j) != i][:25]

    rows = []
    for rank, j in enumerate(top_idx, start=1):
        rows.append({
            'rank': rank,
            'idx': int(j),
            'title': records[j]['title'],
            'artist': records[j]['artist'],
            'genre': records[j]['genre'],
            'filename': records[j]['filename'],
            'cosine_distance': float(cosine_distance[i, j]),
            'cosine_similarity': float(cosine_similarity[i, j]),
            'probability': float(probs[j]),
        })

    nz_probs = probs[probs > 0.0]
    sorted_probs = np.sort(nz_probs)[::-1] if nz_probs.size else np.array([], dtype=np.float64)
    entropy_nats = float(-np.sum(nz_probs * np.log(np.clip(nz_probs, 1e-12, 1.0)))) if nz_probs.size else 0.0
    mass_top_25 = float(np.sum(sorted_probs[:25])) if sorted_probs.size else 0.0

    similarity_payload[str(i)] = {
        'top_25': rows,
        'distribution': {
            'nonzero_count': int(nz_probs.size),
            'max_probability': float(sorted_probs[0]) if sorted_probs.size else 0.0,
            'mean_probability': float(np.mean(nz_probs)) if nz_probs.size else 0.0,
            'median_probability': float(np.median(nz_probs)) if nz_probs.size else 0.0,
            'entropy_nats': entropy_nats,
            'mass_top_1': float(np.sum(sorted_probs[:1])) if sorted_probs.size else 0.0,
            'mass_top_5': float(np.sum(sorted_probs[:5])) if sorted_probs.size else 0.0,
            'mass_top_10': float(np.sum(sorted_probs[:10])) if sorted_probs.size else 0.0,
            'mass_top_25': mass_top_25,
            'mass_remaining': float(max(0.0, 1.0 - mass_top_25)),
        },
    }

sim_json = json.dumps(similarity_payload, ensure_ascii=False)

reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=min(10, max(2, emb.shape[0] - 1)),
    MN_ratio=0.5,
    FP_ratio=1.5,
    random_state=7777,
)
coords = reducer.fit_transform(emb.astype(np.float32))

genres = sorted({r['genre'] for r in records}, key=lambda g: g.lower())
palette = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
    '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#393b79', '#637939',
]
genre_color = {g: palette[i % len(palette)] for i, g in enumerate(genres)}

fig = go.Figure()
for genre in genres:
    pts = [r for r in records if r['genre'] == genre]
    idxs = [p['idx'] for p in pts]
    custom = [[
        p['title'], p['artist'], p['genre'], p['bpm'], p['key'], p['album'],
        p['track_number'], p['filename'], p['snippet_uri'], p['snippet_start'],
        p['snippet_end'], p['snippet_rms'], p['idx'],
    ] for p in pts]

    fig.add_scatter(
        x=coords[idxs, 0],
        y=coords[idxs, 1],
        mode='markers',
        name=genre,
        marker=dict(size=9, color=genre_color[genre], line=dict(color='black', width=0.5)),
        customdata=custom,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Artist: %{customdata[1]}<br>'
            'Genre: %{customdata[2]}<br>'
            'BPM: %{customdata[3]}<br>'
            'Key: %{customdata[4]}<br>'
            'Album: %{customdata[5]}<br>'
            'Track #: %{customdata[6]}<br>'
            'File: %{customdata[7]}<extra></extra>'
        ),
    )

xmin, xmax = float(np.min(coords[:, 0])), float(np.max(coords[:, 0]))
ymin, ymax = float(np.min(coords[:, 1])), float(np.max(coords[:, 1]))
xmid, ymid = (xmin + xmax) / 2.0, (ymin + ymax) / 2.0
xspan0, yspan0 = max(1.0, (xmax - xmin) * 0.55), max(1.0, (ymax - ymin) * 0.55)

zoom_factors = [0.4, 0.7, 1.0, 1.5, 2.2]
zoom_buttons = []
for factor in zoom_factors:
    zoom_buttons.append(dict(
        label=f'Zoom {factor:.1f}x',
        method='relayout',
        args=[{
            'xaxis.range': [xmid - xspan0 * factor, xmid + xspan0 * factor],
            'yaxis.range': [ymid - yspan0 * factor, ymid + yspan0 * factor],
        }],
    ))
zoom_buttons.append(dict(label='Reset', method='relayout', args=[{'xaxis.autorange': True, 'yaxis.autorange': True}]))

fig.update_layout(
    title='Interactive PaCMAP (Click a point: auto-play + top-25 similar)',
    xaxis_title='PaCMAP-1',
    yaxis_title='PaCMAP-2',
    width=1100,
    height=760,
    margin=dict(t=120),
    legend_title_text='Genre',
    updatemenus=[dict(
        type='buttons',
        direction='right',
        showactive=False,
        x=0.0,
        xanchor='left',
        y=1.12,
        yanchor='top',
        buttons=zoom_buttons,
    )],
)

plot_div_id = 'pacmap_click_audio_plot'
plot_html = fig.to_html(include_plotlyjs=True, full_html=False, div_id=plot_div_id)

meta_audio_html = '''
<div style="margin-top:12px;">
  <div id="track-meta" style="padding:8px;border:1px solid #ddd;border-radius:6px;font-size:13px;">
    Click a point to auto-play its high-RMS middle-66% snippet and show top-25 similar songs.
  </div>
  <audio id="track-audio" controls style="width:100%;margin-top:8px;"></audio>
  <div id="similarity-panel" style="margin-top:10px;padding:8px;border:1px solid #ddd;border-radius:6px;font-size:12px;">
    Similarity panel will appear here after you click a point.
  </div>
</div>
'''

script = '''
<script>
(function() {
  const simMap = __SIM_MAP__;
  const plot = document.getElementById('__PLOT_ID__');
  const meta = document.getElementById('track-meta');
  const audio = document.getElementById('track-audio');
  const panel = document.getElementById('similarity-panel');
  if (!plot || !meta || !audio || !panel) return;

  function fmtPct(v) {
    const n = Number(v || 0);
    return (n * 100).toFixed(2) + '%';
  }

  function fmtNum(v, digits) {
    return Number(v || 0).toFixed(digits);
  }

  plot.on('plotly_click', function(ev) {
    if (!ev || !ev.points || !ev.points.length) return;
    const p = ev.points[0];
    const c = p.customdata || [];

    const title = c[0] || 'Unknown title';
    const artist = c[1] || '';
    const genre = c[2] || '';
    const bpm = c[3] || '';
    const key = c[4] || '';
    const album = c[5] || '';
    const trackNum = c[6] || '';
    const filename = c[7] || '';
    const snippetUri = c[8] || '';
    const start = Number(c[9] || 0).toFixed(2);
    const end = Number(c[10] || 0).toFixed(2);
    const snippetRms = Number(c[11] || 0);
    const sourceIdx = String(c[12] || '');

    meta.innerHTML =
      '<b>' + title + '</b>' +
      '<br>Artist: ' + artist +
      '<br>Genre: ' + genre +
      '<br>BPM: ' + bpm + ' | Key: ' + key +
      '<br>Album: ' + album + ' | Track #: ' + trackNum +
      '<br>File: ' + filename +
      '<br>Snippet: ' + start + 's → ' + end + 's' + ' | RMS: ' + fmtNum(snippetRms, 5);

    if (snippetUri) {
      audio.src = snippetUri;
      const playPromise = audio.play();
      if (playPromise && playPromise.catch) {
        playPromise.catch(() => { /* autoplay policy */ });
      }
    } else {
      audio.removeAttribute('src');
      audio.load();
    }

    const simEntry = simMap[sourceIdx] || {};
    const matches = Array.isArray(simEntry) ? simEntry : (simEntry.top_25 || []);
    const dist = Array.isArray(simEntry) ? null : (simEntry.distribution || null);
    let sumProb = 0;
    for (const m of matches) sumProb += Number(m.probability || 0);

    const rows = matches.map(m =>
      '<tr>' +
        '<td style="text-align:right;">' + m.rank + '</td>' +
        '<td>' + (m.title || '') + '</td>' +
        '<td>' + (m.artist || '') + '</td>' +
        '<td>' + (m.genre || '') + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.cosine_distance, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtNum(m.cosine_similarity, 4) + '</td>' +
        '<td style="text-align:right;">' + fmtPct(m.probability) + '</td>' +
      '</tr>'
    ).join('');

    let distHtml = '';
    if (dist) {
      distHtml =
        '<div style="margin-bottom:8px;">' +
        '<b>Similarity Probability Distribution</b><br>' +
        'Top-1: <b>' + fmtPct(dist.mass_top_1) + '</b> | ' +
        'Top-5: <b>' + fmtPct(dist.mass_top_5) + '</b> | ' +
        'Top-10: <b>' + fmtPct(dist.mass_top_10) + '</b> | ' +
        'Top-25: <b>' + fmtPct(dist.mass_top_25) + '</b> | ' +
        'Remaining: <b>' + fmtPct(dist.mass_remaining) + '</b><br>' +
        'Entropy (nats): <b>' + fmtNum(dist.entropy_nats, 4) + '</b> | ' +
        'Active songs: <b>' + Number(dist.nonzero_count || 0) + '</b>' +
        '</div>';
    }

    panel.innerHTML =
      '<div style="margin-bottom:6px;"><b>Top 25 Similar Songs by Cosine Distance</b></div>' +
      '<div style="margin-bottom:6px;">Probability model: softmax(cosine similarity), temperature=0.08. ' +
      'Top-25 cumulative probability: <b>' + fmtPct(sumProb) + '</b></div>' +
      distHtml +
      '<div style="max-height:340px;overflow:auto;border:1px solid #eee;">' +
      '<table style="width:100%;border-collapse:collapse;font-size:12px;">' +
      '<thead><tr>' +
      '<th style="text-align:right;">#</th><th>Title</th><th>Artist</th><th>Genre</th>' +
      '<th style="text-align:right;">CosDist</th><th style="text-align:right;">CosSim</th><th style="text-align:right;">Prob</th>' +
      '</tr></thead><tbody>' + rows + '</tbody></table></div>';
  });
})();
</script>
'''
script = script.replace('__SIM_MAP__', sim_json).replace('__PLOT_ID__', plot_div_id)

interactive_html = '\n'.join([
    '<!doctype html>',
    '<html><head><meta charset="utf-8"></head><body style="font-family:Arial,sans-serif;">',
    plot_html,
    meta_audio_html,
    script,
    '</body></html>',
])

interactive_html_path = INTERACTIVE_HTML_PATH
interactive_html_path.parent.mkdir(parents=True, exist_ok=True)
interactive_html_path.write_text(interactive_html, encoding='utf-8')

missing_snippets = sum(1 for r in records if not r['snippet_uri'])
print(f'Prepared snippets: {len(records) - missing_snippets}/{len(records)} tracks')
print(f'Snippet cache dir: {SNIPPET_CACHE_DIR}')
print('Similarity panel: top 25 by cosine distance with softmax probability distribution (temperature=0.08).')
print(f'Standalone HTML saved to: {interactive_html_path}')
try:
    opened = webbrowser.open_new_tab(interactive_html_path.resolve().as_uri())
    print(f'Opened in browser: {opened}')
except Exception as exc:
    print(f'Could not auto-open browser ({type(exc).__name__}): {exc}')


Prepared snippets: 179/179 tracks
Snippet cache dir: /Users/josephdaher/Git Repositories/djprojectexploration/data/snippets/dataset_tracks
Similarity panel: top 25 by cosine distance with softmax probability distribution (temperature=0.08).
Standalone HTML saved to: /Users/josephdaher/Git Repositories/djprojectexploration/data/exports/dataset_tracks_interactive_pacmap.html


Python(75196) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Opened in browser: True
